In [1]:
import numpy as np
from tqdm import tqdm 

In [4]:
!ls

agents		   eval_zsrl.out	    media
compute_mean.py    eval_zsrl.py		    nohup.out
custom_dmc_tasks   exorl_reformatter.py     proc.out
data_loader.ipynb  inspect_eval_dataset.py  pylintrc
data_loader.py	   LICENSE		    README.md
datasets	   lt_buffer.py		    requirements_org.txt
dmc.py		   main_d4rl.py		    requirements.txt
download.sh	   main_exorl_old.py	    rewards
error.log	   main_exorl.py	    utils.py


In [9]:
data=np.load("./datasets/rl_reward_light0_batch_0.npz")
print(data)

NpzFile './datasets/rl_reward_light0_batch_0.npz' with keys: positions, normals, directions, radiance, ray_ids...


In [ ]:
# data form
print(data["positions"])

[[-0.99807364  0.99999994  0.9953933 ]
 [-0.99999994  0.9999259   0.9967052 ]
 [-0.99825233  0.99999994  0.99617267]
 ...
 [ 0.99958277 -1.          0.9997343 ]
 [-0.99999994  0.01359976  0.7691431 ]
 [ 0.437012   -0.46265358  0.09748062]]


In [ ]:
# shape of data?
# around 44 millions
# (44290706, 3)
data['positions'].shape


In [29]:
positions=data["positions"]
normals=data['normals']
print(normals)

[[ 0.0000000e+00 -1.0000000e+00 -4.3711388e-08]
 [ 1.0000000e+00  0.0000000e+00 -4.3711388e-08]
 [ 0.0000000e+00 -1.0000000e+00 -4.3711388e-08]
 ...
 [ 0.0000000e+00  1.0000000e+00 -4.3711392e-08]
 [ 1.0000000e+00  0.0000000e+00 -4.3711388e-08]
 [ 2.9237166e-01  0.0000000e+00 -9.5630473e-01]]


In [32]:
state=np.concatenate([positions,normals],axis=1)
state.shape[0]

44290706

In [33]:
np.zeros(state.shape[0])

array([0., 0., 0., ..., 0., 0., 0.], shape=(44290706,))

In [ ]:
positions  = data["positions"]     # (N, 3)
normals    = data["normals"]       # (N, 3)
directions = data["directions"]    # (N, 3)
radiance   = data["radiance"]      # (N, 3)
ray_ids    = data["ray_ids"]       # (N,)
bounce_ids = data["bounce_ids"]    # (N,)

# ----------- Build 6D state -----------
s = np.concatenate([positions, normals], axis=1)   # (N, 6)

# ----------- Action (3D) -----------
a = directions.copy()                              # (N, 3)

# ----------- Reward (3D) -----------
r = radiance.copy()                                # (N, 3)

    # Build structured transitions for sorting
N = s.shape[0] # returns the length of states, which is first element of tuple

transitions = np.zeros(N, dtype=[
        ('ray', np.int64),
        ('bounce', np.int64),
        ('s', float, (6,)),
        ('a', float, (3,)),
        ('r', float, (3,))
    ])

In [1]:
from loguru import logger


2026-04-23 12:38:31.042 | INFO     | __main__:<module>:1 - sidvsodv


In [38]:
ray_ids

array([  263172,   263174,   263176, ..., 16514028, 16514036, 16514037],
      shape=(44290706,), dtype=uint32)

In [ ]:


def load_light_transport_npz(file_path):
    """
    Load rl_reward_batch_*.npz and convert into (s, a, r, s_next)
    with:
        state      s      = (position[3], normal[3])         = 6D
        action     a      = direction (wo_world)             = 3D
        reward     r      = radiance (RGB)                   = 3D
        next state s_next = next (pos, normal) OR zero (6D)
    Terminal transitions have:
        s_next = zeros(6)
    """

    data = np.load(file_path)
    breakpoint() # for debugging
    # -------------------------------
    # what is normal vector do?
    # a normal means the surface normal.
    # what is ray_ids?
    # what is bounce ids?
    #--------------------------------
    positions  = data["positions"]     # (N, 3)
    normals    = data["normals"]       # (N, 3)
    directions = data["directions"]    # (N, 3)
    radiance   = data["radiance"]      # (N, 3)
    ray_ids    = data["ray_ids"]       # (N,)
    bounce_ids = data["bounce_ids"]    # (N,)

    # ----------- Build 6D state -----------
    s = np.concatenate([positions, normals], axis=1)   # (N, 6)

    # ----------- Action (3D) -----------
    a = directions.copy()                              # (N, 3)

    # ----------- Reward (3D) -----------
    r = radiance.copy()                                # (N, 3)

    # Build structured transitions for sorting
    N = s.shape[0] # returns the length of states, which is first element of tuple
    transitions = np.zeros(N, dtype=[
        ('ray', np.int64),
        ('bounce', np.int64),
        ('s', float, (6,)),
        ('a', float, (3,)),
        ('r', float, (3,))
    ])

    transitions['ray']    = ray_ids
    transitions['bounce'] = bounce_ids
    transitions['s']      = s
    transitions['a']      = a
    transitions['r']      = r

    # Sort by ray, then bounce
    transitions.sort(order=['ray', 'bounce'])

    # ----------- Build transitions: (s, a, r, s_next) -----------
    s_list = []
    a_list = []
    r_list = []
    s_next_list = []

    for i in tqdm(range(N), desc="proc"):
        s_t = transitions['s'][i]
        a_t = transitions['a'][i]
        r_t = transitions['r'][i]
        ray_t = transitions['ray'][i]
        bounce_t = transitions['bounce'][i]

        # Check if next record is same ray & next bounce
        if i < N - 1:
            ray_next = transitions['ray'][i + 1]
            bounce_next = transitions['bounce'][i + 1]

            if ray_next == ray_t and bounce_next == bounce_t + 1:
                # Valid continuation
                s_next = transitions['s'][i + 1]
            else:
                # Terminal: ray died after this bounce
                s_next = np.zeros(6, dtype=float)
        else:
            # Last record is always terminal
            s_next = np.zeros(6, dtype=float)

        s_list.append(s_t)
        a_list.append(a_t)
        r_list.append(r_t)
        s_next_list.append(s_next)

    # Convert to numpy arrays
    S      = np.array(s_list)       # (N, 6)
    A      = np.array(a_list)       # (N, 3)
    R      = np.array(r_list)       # (N, 3)
    S_next = np.array(s_next_list)  # (N, 6)

    return S, A, R, S_next

if __name__ == "__main__":
    file_path = "./datasets/rl_reward_light0_batch_0.npz"
    s, a, r, s_next = load_light_transport_npz(file_path)

    print("states:", s.shape)
    print("actions:", a.shape)
    print("rewards:", r.shape)
    print("next_states:", s_next.shape)

    print("first state:", s[0])
    print("first action:", a[0])
    print("first reward:", r[0])
    print("first next_state:", s_next[0])


proc: 100%|██████████| 44290706/44290706 [00:38<00:00, 1161864.68it/s]


states: (44290706, 6)
actions: (44290706, 3)
rewards: (44290706, 3)
next_states: (44290706, 6)
first state: [-9.98073637e-01  9.99999940e-01  9.95393276e-01  0.00000000e+00
 -1.00000000e+00 -4.37113883e-08]
first action: [ 0.26037055 -0.27235171 -0.92629993]
first reward: [0.885809   0.69885898 0.66642201]
first next_state: [-0.43719518  0.4131335  -1.          0.          0.          1.        ]
